# RSVP-Admissible AutoML (CPU-Only): GA + Swarm Algorithm Selection

This notebook performs **algorithm selection + hyperparameter search** using **Genetic Algorithms (GA)** and a **discrete PSO-style swarm**, while enforcing an **RSVP-style admissibility filter** (constraint-first selection).

- **Admissibility first:** reject (or penalize) candidates that violate constraints (stability, resource bounds, parsimony).
- **Then optimize:** among admissible candidates, optimize cross-validated performance.

Generated: 2026-01-02 00:34:11


## 0. Setup

In [1]:
import os, math, time, json, random, warnings
from dataclasses import dataclass
from typing import Any, Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.svm import SVC, SVR
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.datasets import load_breast_cancer, load_diabetes, make_classification, make_regression
from sklearn.decomposition import PCA

from scipy.stats import entropy

warnings.filterwarnings("ignore")
random.seed(7)
np.random.seed(7)


## 1. Data Loading

Pick one of:
- built-in datasets
- synthetic data
- custom CSV (set `CSV_PATH` and `TARGET_COL`)


In [2]:
TASK = "classification"      # "classification" or "regression"
DATASET = "diabetes"    # "diabetes", "synthetic", "csv"

CSV_PATH = None              # e.g. "data.csv"
TARGET_COL = None            # e.g. "label"

def load_data(task: str, dataset: str):
    if dataset == "breast_cancer":
        d = load_breast_cancer(as_frame=True)
        return d.data, d.target, "classification"
    if dataset == "diabetes":
        d = load_diabetes(as_frame=True)
        return d.data, d.target, "regression"
    if dataset == "synthetic":
        if task == "classification":
            X, y = make_classification(
                n_samples=1200, n_features=30, n_informative=10, n_redundant=5,
                n_clusters_per_class=2, flip_y=0.03, class_sep=1.1, random_state=7
            )
            return pd.DataFrame(X, columns=[f"x{i}" for i in range(X.shape[1])]), pd.Series(y, name="y"), "classification"
        X, y = make_regression(
            n_samples=1200, n_features=40, n_informative=12, noise=12.0, random_state=7
        )
        return pd.DataFrame(X, columns=[f"x{i}" for i in range(X.shape[1])]), pd.Series(y, name="y"), "regression"
    if dataset == "csv":
        assert CSV_PATH and TARGET_COL, "Set CSV_PATH and TARGET_COL."
        df = pd.read_csv(CSV_PATH)
        y = df[TARGET_COL]
        X = df.drop(columns=[TARGET_COL])
        inferred = "classification" if (y.nunique() <= 20 and pd.api.types.is_integer_dtype(y)) else "regression"
        return X, y, inferred
    raise ValueError("Unknown dataset.")

X, y, inferred = load_data(TASK, DATASET)
TASK = inferred
X.head(), y.head(), TASK


(        age       sex       bmi        bp        s1        s2        s3  \
 0  0.038076  0.050680  0.061696  0.021872 -0.044223 -0.034821 -0.043401   
 1 -0.001882 -0.044642 -0.051474 -0.026328 -0.008449 -0.019163  0.074412   
 2  0.085299  0.050680  0.044451 -0.005671 -0.045599 -0.034194 -0.032356   
 3 -0.089063 -0.044642 -0.011595 -0.036656  0.012191  0.024991 -0.036038   
 4  0.005383 -0.044642 -0.036385  0.021872  0.003935  0.015596  0.008142   
 
          s4        s5        s6  
 0 -0.002592  0.019908 -0.017646  
 1 -0.039493 -0.068330 -0.092204  
 2 -0.002592  0.002864 -0.025930  
 3  0.034309  0.022692 -0.009362  
 4 -0.002592 -0.031991 -0.046641  ,
 0    151.0
 1     75.0
 2    141.0
 3    206.0
 4    135.0
 Name: target, dtype: float64,
 'regression')

## 2. Dataset Characterization (Context for Selection)

We record traits that often correlate with algorithm dominance.


In [3]:
def characterize_dataset(X: pd.DataFrame, y: pd.Series, task: str) -> Dict[str, Any]:
    Xn = X.select_dtypes(include=[np.number]).copy()
    traits = {
        "task": task,
        "n_samples": int(X.shape[0]),
        "n_features": int(X.shape[1]),
        "n_numeric": int(Xn.shape[1]),
        "n_categorical": int(X.shape[1] - Xn.shape[1]),
    }

    if Xn.shape[1] > 1:
        C = np.corrcoef(Xn.to_numpy().T)
        C = np.nan_to_num(C, nan=0.0)
        traits["mean_abs_corr"] = float(np.mean(np.abs(C)))
        traits["mean_var"] = float(np.mean(np.var(Xn.to_numpy(), axis=0)))
    else:
        traits["mean_abs_corr"] = float("nan")
        traits["mean_var"] = float("nan")

    if task == "classification":
        counts = np.bincount(pd.Series(y).astype(int).to_numpy())
        p = counts / max(1, counts.sum())
        traits["n_classes"] = int(len(counts))
        traits["class_balance_min"] = float(p.min())
        traits["y_entropy"] = float(entropy(p + 1e-12))
    else:
        yy = pd.Series(y).to_numpy().astype(float)
        traits["y_mean"] = float(np.mean(yy))
        traits["y_std"] = float(np.std(yy))

    if Xn.shape[1] >= 2:
        Z = np.nan_to_num(Xn.to_numpy(), nan=0.0)
        pca = PCA(n_components=min(10, Z.shape[1]), random_state=7)
        pca.fit(Z)
        evr = pca.explained_variance_ratio_
        traits["pca_90_dims"] = int(np.searchsorted(np.cumsum(evr), 0.90) + 1)
    else:
        traits["pca_90_dims"] = None

    return traits

traits = characterize_dataset(X, y, TASK)
pd.Series(traits)


task             regression
n_samples               442
n_features               10
n_numeric                10
n_categorical             0
mean_abs_corr      0.389579
mean_var         0.00226244
y_mean              152.133
y_std               77.0057
pca_90_dims               7
dtype: object

## 3. Preprocessing Pipeline Factory

Preprocessing is part of the genome (`scaler` choice).


In [4]:
def make_preprocessor(X: pd.DataFrame, scaler: str):
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = [c for c in X.columns if c not in numeric_cols]

    if scaler == "standard":
        scaler_obj = StandardScaler()
    elif scaler == "minmax":
        scaler_obj = MinMaxScaler()
    else:
        scaler_obj = "passthrough"

    numeric_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", scaler_obj),
    ])

    cat_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("oh", OneHotEncoder(handle_unknown="ignore", sparse=False)),
    ])

    transformers = []
    if numeric_cols:
        transformers.append(("num", numeric_pipe, numeric_cols))
    if cat_cols:
        transformers.append(("cat", cat_pipe, cat_cols))

    return ColumnTransformer(transformers=transformers, remainder="drop")

def make_pipeline(model, X: pd.DataFrame, scaler: str):
    return Pipeline(steps=[("pre", make_preprocessor(X, scaler)), ("model", model)])


## 4. Model Families & Search Space

In [5]:
MODEL_FAMILIES_CLASSIFICATION = ["logreg", "svm", "rf", "gb", "knn"]
MODEL_FAMILIES_REGRESSION = ["ridge", "svr", "rf", "gb", "knn"]

SCALERS = ["none", "standard", "minmax"]

SPACE = {
    "model_family": MODEL_FAMILIES_CLASSIFICATION if TASK == "classification" else MODEL_FAMILIES_REGRESSION,
    "scaler": SCALERS,

    # classification
    "logreg_C": (1e-3, 30.0),
    "svm_C": (1e-3, 30.0),
    "svm_gamma": (1e-4, 2.0),
    "rf_estimators": (50, 350),
    "rf_max_depth": (2, 30),
    "rf_min_samples_leaf": (1, 10),
    "gb_estimators": (50, 350),
    "gb_lr": (1e-3, 0.3),
    "gb_max_depth": (2, 6),
    "knn_k": (3, 60),
    "knn_weights": ["uniform", "distance"],

    # regression
    "ridge_alpha": (1e-4, 200.0),
    "svr_C": (1e-3, 30.0),
    "svr_gamma": (1e-4, 2.0),
}
SPACE["model_family"]


['ridge', 'svr', 'rf', 'gb', 'knn']

## 5. RSVP Admissibility (Constraint-First)

Admissibility enforces:
- CPU feasibility (cost proxy)
- Parsimony (capacity vs sample size)
- Robustness (CV std bound)

Tune thresholds in `AdmissibilityConfig`.


In [6]:
@dataclass
class AdmissibilityConfig:
    max_cost_units: float = 2.5e7
    max_complexity_ratio: float = 0.25
    max_cv_std: float = 0.08
    hard_reject: bool = True

ADM = AdmissibilityConfig()

def estimate_cost_units(genome: Dict[str, Any], traits: Dict[str, Any]) -> float:
    n = traits["n_samples"]
    d = max(1, traits["n_features"])
    fam = genome["model_family"]
    if fam in ["logreg", "ridge"]:
        return 5e2 * n * d
    if fam in ["svm", "svr"]:
        return 2e3 * (n**2) * 0.02
    if fam == "rf":
        t = genome.get("rf_estimators", 150)
        depth = genome.get("rf_max_depth", 12)
        return 1e2 * t * n * depth
    if fam == "gb":
        t = genome.get("gb_estimators", 150)
        depth = genome.get("gb_max_depth", 3)
        return 1e2 * t * n * depth * 0.8
    if fam == "knn":
        k = genome.get("knn_k", 15)
        return 2e2 * n * d + 1e2 * n * k
    return 1e9

def estimate_complexity_units(genome: Dict[str, Any], traits: Dict[str, Any]) -> float:
    fam = genome["model_family"]
    d = max(1, traits["n_features"])
    if fam in ["logreg", "ridge"]:
        return 1.0 * d
    if fam in ["svm", "svr"]:
        return 5.0 * d
    if fam == "rf":
        t = genome.get("rf_estimators", 150)
        depth = genome.get("rf_max_depth", 12)
        return 0.5 * t * depth
    if fam == "gb":
        t = genome.get("gb_estimators", 150)
        depth = genome.get("gb_max_depth", 3)
        return 0.8 * t * depth
    if fam == "knn":
        k = genome.get("knn_k", 15)
        return 0.3 * k
    return 1e9

def admissible_pre_eval(genome: Dict[str, Any], traits: Dict[str, Any], adm: AdmissibilityConfig) -> Tuple[bool, Dict[str, Any]]:
    info = {}
    cost = estimate_cost_units(genome, traits)
    complexity = estimate_complexity_units(genome, traits)
    info["cost_units"] = float(cost)
    info["complexity_units"] = float(complexity)

    if cost > adm.max_cost_units:
        info["reason"] = "resource_cost_exceeded"
        return (False, info) if adm.hard_reject else (True, info)
    if complexity > adm.max_complexity_ratio * traits["n_samples"]:
        info["reason"] = "complexity_exceeded"
        return (False, info) if adm.hard_reject else (True, info)

    return True, info

def admissible_post_eval(cv_std: float, adm: AdmissibilityConfig) -> Tuple[bool, str]:
    if cv_std > adm.max_cv_std:
        return (False, "cv_instability") if adm.hard_reject else (True, "cv_instability_penalized")
    return True, "ok"


## 6. Genome Sampling, Decoding, Evaluation

In [7]:
def log_uniform(lo, hi):
    return float(np.exp(np.random.uniform(np.log(lo), np.log(hi))))

def sample_genome(task: str) -> Dict[str, Any]:
    fam = random.choice(SPACE["model_family"])
    g = {"model_family": fam, "scaler": random.choice(SPACE["scaler"])}

    if task == "classification":
        if fam == "logreg":
            g["logreg_C"] = log_uniform(*SPACE["logreg_C"])
        elif fam == "svm":
            g["svm_C"] = log_uniform(*SPACE["svm_C"])
            g["svm_gamma"] = log_uniform(*SPACE["svm_gamma"])
        elif fam == "rf":
            g["rf_estimators"] = int(np.random.randint(*SPACE["rf_estimators"]))
            g["rf_max_depth"] = int(np.random.randint(*SPACE["rf_max_depth"]))
            g["rf_min_samples_leaf"] = int(np.random.randint(*SPACE["rf_min_samples_leaf"]))
        elif fam == "gb":
            g["gb_estimators"] = int(np.random.randint(*SPACE["gb_estimators"]))
            g["gb_lr"] = log_uniform(*SPACE["gb_lr"])
            g["gb_max_depth"] = int(np.random.randint(*SPACE["gb_max_depth"]))
        elif fam == "knn":
            g["knn_k"] = int(np.random.randint(*SPACE["knn_k"]))
            g["knn_weights"] = random.choice(SPACE["knn_weights"])
    else:
        if fam == "ridge":
            g["ridge_alpha"] = log_uniform(*SPACE["ridge_alpha"])
        elif fam == "svr":
            g["svr_C"] = log_uniform(*SPACE["svr_C"])
            g["svr_gamma"] = log_uniform(*SPACE["svr_gamma"])
        elif fam == "rf":
            g["rf_estimators"] = int(np.random.randint(*SPACE["rf_estimators"]))
            g["rf_max_depth"] = int(np.random.randint(*SPACE["rf_max_depth"]))
            g["rf_min_samples_leaf"] = int(np.random.randint(*SPACE["rf_min_samples_leaf"]))
        elif fam == "gb":
            g["gb_estimators"] = int(np.random.randint(*SPACE["gb_estimators"]))
            g["gb_lr"] = log_uniform(*SPACE["gb_lr"])
            g["gb_max_depth"] = int(np.random.randint(*SPACE["gb_max_depth"]))
        elif fam == "knn":
            g["knn_k"] = int(np.random.randint(*SPACE["knn_k"]))
            g["knn_weights"] = random.choice(SPACE["knn_weights"])
    return g

def build_model_from_genome(genome: Dict[str, Any], task: str):
    fam = genome["model_family"]
    if task == "classification":
        if fam == "logreg":
            return LogisticRegression(C=genome["logreg_C"], max_iter=500, solver="lbfgs")
        if fam == "svm":
            return SVC(C=genome["svm_C"], gamma=genome["svm_gamma"], kernel="rbf")
        if fam == "rf":
            return RandomForestClassifier(
                n_estimators=genome["rf_estimators"],
                max_depth=genome["rf_max_depth"],
                min_samples_leaf=genome["rf_min_samples_leaf"],
                n_jobs=-1, random_state=7
            )
        if fam == "gb":
            return GradientBoostingClassifier(
                n_estimators=genome["gb_estimators"],
                learning_rate=genome["gb_lr"],
                max_depth=genome["gb_max_depth"],
                random_state=7
            )
        if fam == "knn":
            return KNeighborsClassifier(n_neighbors=genome["knn_k"], weights=genome["knn_weights"])
    else:
        if fam == "ridge":
            return Ridge(alpha=genome["ridge_alpha"], random_state=7)
        if fam == "svr":
            return SVR(C=genome["svr_C"], gamma=genome["svr_gamma"], kernel="rbf")
        if fam == "rf":
            return RandomForestRegressor(
                n_estimators=genome["rf_estimators"],
                max_depth=genome["rf_max_depth"],
                min_samples_leaf=genome["rf_min_samples_leaf"],
                n_jobs=-1, random_state=7
            )
        if fam == "gb":
            return GradientBoostingRegressor(
                n_estimators=genome["gb_estimators"],
                learning_rate=genome["gb_lr"],
                max_depth=genome["gb_max_depth"],
                random_state=7
            )
        if fam == "knn":
            return KNeighborsRegressor(n_neighbors=genome["knn_k"], weights=genome["knn_weights"])
    raise ValueError("Unknown family/task.")

def metric_name(task: str):
    return "f1_macro" if task == "classification" else "neg_mean_squared_error"

def evaluate_genome(genome: Dict[str, Any], X: pd.DataFrame, y: pd.Series, task: str, traits: Dict[str, Any], adm: AdmissibilityConfig, cv_folds: int = 5) -> Dict[str, Any]:
    ok, info = admissible_pre_eval(genome, traits, adm)
    if not ok:
        return {"admissible": False, "reason": info.get("reason","pre_reject"), "score": -np.inf, "cv_mean": None, "cv_std": None, **info}

    model = build_model_from_genome(genome, task)
    pipe = make_pipeline(model, X, genome["scaler"])

    cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=7) if task == "classification" else KFold(n_splits=cv_folds, shuffle=True, random_state=7)
    scores = cross_val_score(pipe, X, y, cv=cv, scoring=metric_name(task), n_jobs=1)

    if task == "classification":
        cv_mean, cv_std = float(np.mean(scores)), float(np.std(scores))
        ok2, why = admissible_post_eval(cv_std, adm)
        if not ok2:
            return {"admissible": False, "reason": why, "score": -np.inf, "cv_mean": cv_mean, "cv_std": cv_std, **info}
        score = cv_mean - 0.15 * cv_std
        return {"admissible": True, "reason": "ok", "score": float(score), "cv_mean": cv_mean, "cv_std": cv_std, **info}

    mse = float(-np.mean(scores))
    rmse = float(np.sqrt(max(mse, 1e-12)))
    rmse_std_proxy = float(np.std(-scores)) / max(rmse, 1e-12) * 0.5
    ok2, why = admissible_post_eval(rmse_std_proxy, adm)
    if not ok2:
        return {"admissible": False, "reason": why, "score": -np.inf, "cv_mean": -rmse, "cv_std": rmse_std_proxy, "rmse": rmse, **info}
    score = (-rmse) - 0.15 * rmse_std_proxy
    return {"admissible": True, "reason": "ok", "score": float(score), "cv_mean": -rmse, "cv_std": rmse_std_proxy, "rmse": rmse, **info}


## 7. Baselines

In [8]:
def baseline_genomes(task: str):
    if task == "classification":
        return [
            {"model_family":"logreg","scaler":"standard","logreg_C":1.0},
            {"model_family":"rf","scaler":"none","rf_estimators":200,"rf_max_depth":12,"rf_min_samples_leaf":1},
            {"model_family":"svm","scaler":"standard","svm_C":1.0,"svm_gamma":0.1},
            {"model_family":"knn","scaler":"standard","knn_k":15,"knn_weights":"distance"},
            {"model_family":"gb","scaler":"none","gb_estimators":200,"gb_lr":0.05,"gb_max_depth":3},
        ]
    return [
        {"model_family":"ridge","scaler":"standard","ridge_alpha":1.0},
        {"model_family":"rf","scaler":"none","rf_estimators":200,"rf_max_depth":12,"rf_min_samples_leaf":1},
        {"model_family":"svr","scaler":"standard","svr_C":5.0,"svr_gamma":0.1},
        {"model_family":"knn","scaler":"standard","knn_k":15,"knn_weights":"distance"},
        {"model_family":"gb","scaler":"none","gb_estimators":200,"gb_lr":0.05,"gb_max_depth":3},
    ]

rows = []
for g in baseline_genomes(TASK):
    r = evaluate_genome(g, X, y, TASK, traits, ADM, cv_folds=5)
    rows.append({**g, **r})

baseline_df = pd.DataFrame(rows).sort_values("score", ascending=False)
baseline_df


,model_family,scaler,ridge_alpha,admissible,reason,score,cv_mean,cv_std,rmse,cost_units,...,rf_estimators,rf_max_depth,rf_min_samples_leaf,svr_C,svr_gamma,knn_k,knn_weights,gb_estimators,gb_lr,gb_max_depth
0,ridge,standard,1.0,False,cv_instability,-inf,-54.982892,2.604521,54.982892,2210000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,rf,none,NaN,False,resource_cost_exceeded,-inf,NaN,NaN,NaN,106080000.0,...,200.0,12.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,svr,standard,NaN,False,cv_instability,-inf,-58.660497,5.291259,58.660497,7814560.0,...,NaN,NaN,NaN,5.0,0.1,NaN,NaN,NaN,NaN,NaN
3,knn,standard,NaN,False,cv_instability,-inf,-57.152427,4.400569,57.152427,1547000.0,...,NaN,NaN,NaN,NaN,NaN,15.0,distance,NaN,NaN,NaN
4,gb,none,NaN,False,complexity_exceeded,-inf,NaN,NaN,NaN,21216000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,200.0,0.05,3.0


## 8. Genetic Algorithm (GA)

In [ ]:
from multiprocessing import Pool, cpu_count

@dataclass
class GAConfig:
    pop_size: int = 16
    generations: int = 5
    elite_frac: float = 0.25
    mutation_rate: float = 0.12
    cv_folds: int = 3
    n_jobs_eval: int = max(1, cpu_count() - 1)

GA = GAConfig()

def mutate_genome(g: Dict[str, Any], rate: float, task: str) -> Dict[str, Any]:
    g2 = dict(g)
    if np.random.rand() < rate * 0.35:
        return sample_genome(task)

    if np.random.rand() < rate:
        g2["scaler"] = random.choice(SPACE["scaler"])

    fam = g2["model_family"]

    def jitter_log(key, lo, hi):
        x = float(g2.get(key, log_uniform(lo, hi)))
        x *= float(np.exp(np.random.normal(0.0, 0.35)))
        g2[key] = float(np.clip(x, lo, hi))

    def jitter_int(key, lo, hi):
        x = int(g2.get(key, int((lo+hi)//2)))
        x += int(np.random.normal(0.0, (hi-lo) * 0.06))
        g2[key] = int(np.clip(x, lo, hi-1))

    if task == "classification":
        if fam == "logreg" and np.random.rand() < rate:
            jitter_log("logreg_C", *SPACE["logreg_C"])
        if fam == "svm":
            if np.random.rand() < rate: jitter_log("svm_C", *SPACE["svm_C"])
            if np.random.rand() < rate: jitter_log("svm_gamma", *SPACE["svm_gamma"])
        if fam == "rf":
            if np.random.rand() < rate: jitter_int("rf_estimators", *SPACE["rf_estimators"])
            if np.random.rand() < rate: jitter_int("rf_max_depth", *SPACE["rf_max_depth"])
            if np.random.rand() < rate: jitter_int("rf_min_samples_leaf", *SPACE["rf_min_samples_leaf"])
        if fam == "gb":
            if np.random.rand() < rate: jitter_int("gb_estimators", *SPACE["gb_estimators"])
            if np.random.rand() < rate: jitter_log("gb_lr", *SPACE["gb_lr"])
            if np.random.rand() < rate: jitter_int("gb_max_depth", *SPACE["gb_max_depth"])
        if fam == "knn":
            if np.random.rand() < rate: jitter_int("knn_k", *SPACE["knn_k"])
            if np.random.rand() < rate: g2["knn_weights"] = random.choice(SPACE["knn_weights"])
    else:
        if fam == "ridge" and np.random.rand() < rate:
            jitter_log("ridge_alpha", *SPACE["ridge_alpha"])
        if fam == "svr":
            if np.random.rand() < rate: jitter_log("svr_C", *SPACE["svr_C"])
            if np.random.rand() < rate: jitter_log("svr_gamma", *SPACE["svr_gamma"])
        if fam == "rf":
            if np.random.rand() < rate: jitter_int("rf_estimators", *SPACE["rf_estimators"])
            if np.random.rand() < rate: jitter_int("rf_max_depth", *SPACE["rf_max_depth"])
            if np.random.rand() < rate: jitter_int("rf_min_samples_leaf", *SPACE["rf_min_samples_leaf"])
        if fam == "gb":
            if np.random.rand() < rate: jitter_int("gb_estimators", *SPACE["gb_estimators"])
            if np.random.rand() < rate: jitter_log("gb_lr", *SPACE["gb_lr"])
            if np.random.rand() < rate: jitter_int("gb_max_depth", *SPACE["gb_max_depth"])
        if fam == "knn":
            if np.random.rand() < rate: jitter_int("knn_k", *SPACE["knn_k"])
            if np.random.rand() < rate: g2["knn_weights"] = random.choice(SPACE["knn_weights"])

    return g2

def crossover(a: Dict[str, Any], b: Dict[str, Any]) -> Dict[str, Any]:
    child = {}
    keys = set(a.keys()) | set(b.keys())
    for k in keys:
        child[k] = a.get(k) if np.random.rand() < 0.5 else b.get(k)
    if "model_family" not in child or child["model_family"] not in SPACE["model_family"]:
        child["model_family"] = a["model_family"]
    if "scaler" not in child or child["scaler"] not in SPACE["scaler"]:
        child["scaler"] = a.get("scaler","none")
    return child

def _eval_one(args):
    g, X, y, task, traits, adm, cv = args
    r = evaluate_genome(g, X, y, task, traits, adm, cv_folds=cv)
    return {**g, **r}
def run_ga(X, y, task: str, traits: Dict[str, Any],
           adm: AdmissibilityConfig, cfg: GAConfig):

    pop = [sample_genome(task) for _ in range(cfg.pop_size)]
    frames = []

    for gen in range(cfg.generations):
        print(f"\nGA gen {gen:02d} | starting evaluation ({cfg.pop_size} genomes)")
        t0 = time.time()

        # --- parallel evaluation (THIS must be inside the loop) ---
        with Pool(processes=cfg.n_jobs_eval) as pool:
            results = pool.map(
                _eval_one,
                [(g, X, y, task, traits, adm, cfg.cv_folds) for g in pop]
            )

        print(f"GA gen {gen:02d} | evaluation done in {time.time() - t0:.1f}s")

        df = pd.DataFrame(results)
        df["generation"] = gen
        frames.append(df)

        # --- selection ---
        admiss = df[df["admissible"] == True]
        pool_df = admiss if len(admiss) else df
        pool_df = pool_df.sort_values("score", ascending=False)

        elite_n = max(2, int(cfg.elite_frac * len(pool_df)))
        elites = pool_df.head(elite_n).to_dict(orient="records")

        best = pool_df.iloc[0]
        print(
            f"GA gen {gen:02d} | "
            f"best={best['score']:.4f} | "
            f"fam={best['model_family']} | "
            f"time={time.time() - t0:.1f}s"
        )

        # --- reproduction ---
        next_pop = []
        keep_n = max(1, elite_n // 3)

        for e in elites[:keep_n]:
            gk = {
                k: e[k] for k in e.keys()
                if (
                    k in SPACE
                    or k.endswith(("_C", "_gamma", "_alpha"))
                    or k.startswith(("rf_", "gb_", "knn_"))
                    or k in ["model_family", "scaler"]
                )
            }
            next_pop.append(gk)

        while len(next_pop) < cfg.pop_size:
            p1, p2 = random.choice(elites), random.choice(elites)
            child = mutate_genome(
                crossover(p1, p2),
                cfg.mutation_rate,
                task
            )
            next_pop.append(child)

        pop = next_pop[:cfg.pop_size]

    return pd.concat(frames, ignore_index=True)

ga_df = run_ga(X, y, TASK, traits, ADM, GA)
ga_df.sort_values("score", ascending=False).head(10)


## 9. Discrete Swarm (PSO-style)

In [ ]:
@dataclass
class PSOConfig:
    n_particles: int = 25
    iters: int = 18
    inertia: float = 0.55
    c1: float = 1.2
    c2: float = 1.2
    cv_folds: int = 5
    n_jobs_eval: int = max(1, cpu_count() - 1)

PSO = PSOConfig()

GENE_KEYS = [
    "model_family","scaler",
    "logreg_C","svm_C","svm_gamma","ridge_alpha","svr_C","svr_gamma",
    "rf_estimators","rf_max_depth","rf_min_samples_leaf",
    "gb_estimators","gb_lr","gb_max_depth",
    "knn_k","knn_weights"
]

def pso_step(particles, velocities, pbest, gbest, cfg: PSOConfig, task: str):
    new_particles, new_vel = [], []
    for g, v, pb in zip(particles, velocities, pbest):
        g2, v2 = dict(g), dict(v)
        for k in GENE_KEYS:
            vk = float(v2.get(k, 0.05))
            vk = cfg.inertia * vk
            if np.random.rand() < cfg.c1 * 0.25:
                vk += 0.10
            if np.random.rand() < cfg.c2 * 0.25:
                vk += 0.10
            vk = float(np.clip(vk, 0.0, 0.6))
            v2[k] = vk

            r = np.random.rand()
            if r < vk * 0.5 and k in pb:
                g2[k] = pb[k]
            elif r < vk and k in gbest:
                g2[k] = gbest[k]

        if np.random.rand() < 0.25:
            g2 = mutate_genome(g2, rate=0.08, task=task)

        if g2.get("model_family") not in SPACE["model_family"]:
            g2 = sample_genome(task)

        new_particles.append(g2)
        new_vel.append(v2)
    return new_particles, new_vel

def run_pso(X, y, task: str, traits: Dict[str, Any], adm: AdmissibilityConfig, cfg: PSOConfig, seed_genomes: Optional[List[Dict[str,Any]]] = None):
    if seed_genomes:
        particles = [dict(seed_genomes[i % len(seed_genomes)]) for i in range(cfg.n_particles)]
    else:
        particles = [sample_genome(task) for _ in range(cfg.n_particles)]

    velocities = [dict() for _ in range(cfg.n_particles)]

    def eval_particles(parts):
        with Pool(processes=cfg.n_jobs_eval) as pool:
            results = pool.map(_eval_one, [(g, X, y, task, traits, adm, cfg.cv_folds) for g in parts])
        return pd.DataFrame(results)

    df0 = eval_particles(particles)
    pbest = particles.copy()
    pbest_scores = df0["score"].to_numpy()
    gbest_idx = int(np.argmax(pbest_scores))
    gbest = dict(particles[gbest_idx])
    gbest_score = float(pbest_scores[gbest_idx])

    hist = [df0.assign(iteration=0)]
    print(f"PSO iter 00 | best={gbest_score:.4f} | fam={gbest['model_family']}")

    for it in range(1, cfg.iters + 1):
        particles, velocities = pso_step(particles, velocities, pbest, gbest, cfg, task)
        df = eval_particles(particles)
        hist.append(df.assign(iteration=it))

        scores = df["score"].to_numpy()
        improved = scores > pbest_scores
        for i, ok in enumerate(improved):
            if ok:
                pbest[i] = dict(particles[i])
                pbest_scores[i] = scores[i]

        best_idx = int(np.argmax(pbest_scores))
        if float(pbest_scores[best_idx]) > gbest_score:
            gbest_score = float(pbest_scores[best_idx])
            gbest = dict(pbest[best_idx])

        print(f"PSO iter {it:02d} | best={gbest_score:.4f} | fam={gbest['model_family']}")

    return pd.concat(hist, ignore_index=True), gbest, gbest_score

ga_top = ga_df[ga_df["admissible"]==True].sort_values("score", ascending=False).head(8)
seed = ga_top.to_dict(orient="records")
pso_df, pso_best, pso_best_score = run_pso(X, y, TASK, traits, ADM, PSO, seed_genomes=seed)
pso_best, pso_best_score


## 10. Results + Visualizations (Diagnostics)

Plots:
- Best score trajectory (GA, PSO)
- Family frequency among admissible candidates
- Score vs estimated cost/complexity


In [ ]:
def top_k(df, k=10):
    return df.sort_values("score", ascending=False).head(k)

print("Top GA (admissible):")
display(top_k(ga_df[ga_df["admissible"]==True], 10))

print("Top PSO (admissible):")
display(top_k(pso_df[pso_df["admissible"]==True], 10))

ga_adm = ga_df[ga_df["admissible"]==True].copy()
pso_adm = pso_df[pso_df["admissible"]==True].copy()

# Trajectories
plt.figure()
plt.plot(ga_adm.groupby("generation")["score"].max().index, ga_adm.groupby("generation")["score"].max().values)
plt.xlabel("GA generation"); plt.ylabel("Best RSVP score"); plt.title("GA Best Score Trajectory")
plt.show()

plt.figure()
plt.plot(pso_adm.groupby("iteration")["score"].max().index, pso_adm.groupby("iteration")["score"].max().values)
plt.xlabel("PSO iteration"); plt.ylabel("Best RSVP score"); plt.title("PSO Best Score Trajectory")
plt.show()

# Family frequency
plt.figure()
ga_adm["model_family"].value_counts().plot(kind="bar")
plt.xlabel("Model family"); plt.ylabel("Count"); plt.title("GA Admissible Family Frequency")
plt.show()

plt.figure()
pso_adm["model_family"].value_counts().plot(kind="bar")
plt.xlabel("Model family"); plt.ylabel("Count"); plt.title("PSO Admissible Family Frequency")
plt.show()

# Score vs cost/complexity
plt.figure()
plt.scatter(ga_df["complexity_units"], ga_df["score"])
plt.xlabel("Estimated complexity"); plt.ylabel("RSVP score"); plt.title("GA: Score vs Complexity")
plt.show()

plt.figure()
plt.scatter(ga_df["cost_units"], ga_df["score"])
plt.xlabel("Estimated cost"); plt.ylabel("RSVP score"); plt.title("GA: Score vs Cost")
plt.show()


## 11. Export Best Admissible Genome

In [ ]:
best_ga = ga_df[ga_df["admissible"]==True].sort_values("score", ascending=False).head(1)
best_pso = pso_df[pso_df["admissible"]==True].sort_values("score", ascending=False).head(1)

best = pd.concat([best_ga, best_pso], ignore_index=True).sort_values("score", ascending=False).head(1)
best_row = best.iloc[0].to_dict()

best_genome = {k: best_row.get(k) for k in GENE_KEYS if best_row.get(k) is not None}
print(json.dumps(best_genome, indent=2))

os.makedirs("artifacts", exist_ok=True)
with open("artifacts/best_genome.json", "w") as f:
    f.write(json.dumps(best_genome, indent=2))

print("Saved to artifacts/best_genome.json")
